# CSCIE89 HW05 – Problem 2: Batch Normalization in the flower CNN

This notebook solves Problem 2. It starts from **1_Augmentation_CNN_Classify_flowers** and keeps only the parts needed for this problem. The additions are:

- A new network, `BatchNormCNN`: the same `BasicCNN` with a **Batch Normalization layer (`nn.BatchNorm2d`) after each `Conv2d` layer**.
- **Overall precision** measured on the **entire test set** (all 551 test images).
- Google Drive + local Colab setup, so the dataset is downloaded only once.
- The four combinations run one after another in this notebook, so one **Run all** gives every result:

| Run | Network | Augmentation |
|---|---|---|
| 1 | Original `BasicCNN` | No |
| 2 | Original `BasicCNN` | Yes |
| 3 | `BatchNormCNN` | No |
| 4 | `BatchNormCNN` | Yes |

The image size (64 × 64), Adam optimizer, learning rate, number of epochs, batch size, and data split are the same in every run.

## 1. Install packages and import tools

If the imports fail, uncomment the installation line, run it once, then restart the session. In Colab use **Runtime → Change runtime type → GPU**.

In [ ]:
# %pip install torch torchvision scikit-learn matplotlib pillow   # Uncomment once if the packages are missing.

from pathlib import Path                         # Path builds file paths that work on any OS.
import copy                                      # deepcopy keeps a frozen copy of the best weights.
import random                                    # Python's random-number generator (seeded below).
import time                                      # Measure how long training takes.
import numpy as np                               # Arrays and numeric helpers.
import matplotlib.pyplot as plt                  # Plotting library for charts.
from PIL import Image                            # Pillow opens image files.
import torch                                     # PyTorch: tensors and training tools.
from torch import nn                             # Neural-network layers (Conv2d, BatchNorm2d, Linear, ...).
from torch.utils.data import Dataset, DataLoader # Dataset loads one photo; DataLoader groups photos into batches.
from torchvision import datasets, transforms     # ImageFolder dataset reader and image transformations.
from torchvision.datasets.utils import download_and_extract_archive  # Download and unpack the dataset archive.
from sklearn.model_selection import train_test_split  # Reproducible, stratified data split.
from sklearn.metrics import accuracy_score, classification_report, precision_score  # Accuracy, per-class report, precision.

# Seeds make the split and initial weights repeatable, but do not guarantee
# identical floating-point results across all devices or PyTorch versions.
SEED = 42                                        # One seed used everywhere.
random.seed(SEED)                                # Seed Python's random module.
np.random.seed(SEED)                             # Seed NumPy.
torch.manual_seed(SEED)                          # Seed PyTorch (weight initialization, augmentation).
torch.set_num_threads(4)  # Avoid excessive CPU thread overhead on this small model.
device = torch.device('cuda' if torch.cuda.is_available() else          # Use an NVIDIA GPU if present,
                      'mps' if torch.backends.mps.is_available() else 'cpu')  # else Apple GPU, else the CPU.
IMAGE_SIZE = 64                                  # Every photo is resized to 64 x 64 pixels.
BATCH_SIZE = 64                                  # Photos per training step.
EPOCHS = 30                                      # Complete passes through the training data.
LEARNING_RATE = 0.001                            # Adam step size.
print('PyTorch:', torch.__version__, '| Device:', device)  # Show the PyTorch version and the device in use.

## 2. Google Drive + local Colab setup, then split the data

The [Flower Photos dataset](https://www.tensorflow.org/tutorials/images/classification) has 3,670 photographs in five folders: daisy, dandelion, roses, sunflowers, and tulips (about 220 MB).

- A **permanent copy** of the photos is kept in Google Drive (`DRIVE_DATA_DIR`).
- Training reads the photos from Colab's **fast local disk** (`DATA_DIR = /content/data`), because reading thousands of small files directly from Drive is slow.
- The first time, the dataset is downloaded into Colab and copied to Drive. Later sessions copy it from Drive instead of downloading it again.

`ImageFolder` assigns labels from the alphabetically sorted folder names. We make a reproducible stratified 70% / 15% / 15% split into training, validation, and test sets.

In [ ]:
# GOOGLE DRIVE + LOCAL COLAB SETUP

from google.colab import drive                   # Colab tool to connect your Google Drive.
import shutil                                    # Copy whole folders.

drive.mount('/content/drive')                    # Mount Google Drive (asks for permission the first time).

# Permanent location in Google Drive
PROJECT_DIR = Path('/content/drive/MyDrive/AI/Flowers_CNN')  # Project folder inside your Drive.
DRIVE_DATA_DIR = PROJECT_DIR / 'data'            # Permanent data folder in Drive.
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)  # Create it; no error if it already exists.

# IMPORTANT: # DATA_DIR is the fast temporary Colab drive

DATA_DIR = Path('/content/data')                 # Fast local folder; erased when the Colab session ends.
DATA_DIR.mkdir(parents=True, exist_ok=True)      # Create it; no error if it already exists.


flower_root = DATA_DIR / 'flower_photos'         # Flowers on the fast local disk (used for training).
drive_flower_root = DRIVE_DATA_DIR / 'flower_photos'  # Permanent copy of the flowers in Drive.

# If flowers are not already in Colab
if not flower_root.exists():

    # Copy from Google Drive if already saved there
    if drive_flower_root.exists():
        shutil.copytree(drive_flower_root, flower_root)  # Drive -> local disk.

    # Otherwise download them into Colab
    else:
        download_and_extract_archive(            # Download the archive and unpack it:
            'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',  # dataset URL,
            download_root=str(DATA_DIR),         # into the local folder.
        )

# Save a permanent copy to Google Drive if one does not exist
if not drive_flower_root.exists():
    shutil.copytree(flower_root, drive_flower_root)  # Local disk -> Drive (only once).

# ______________________________________________________________________________________________________
catalog = datasets.ImageFolder(flower_root)      # Finds every photo; the folder name is the label.
class_names = catalog.classes                    # Class names in label order.

# Each record is (path_to_image, integer_class_label).
records = [(Path(path), label) for path, label in catalog.samples]  # List of all photos with their labels.

labels = [label for _, label in records]         # Just the labels, used to stratify the split.

train_records, remaining = train_test_split(     # First split: 70% training, 30% set aside.
    records,                                     # All photos.
    test_size=0.30,                              # 30% set aside for validation + test.
    random_state=SEED,                           # Repeatable split.
    stratify=labels                              # Keep class proportions equal in both parts.
)

val_records, test_records = train_test_split(    # Second split: the 30% becomes 15% validation + 15% test.
    remaining,                                   # The photos set aside above.
    test_size=0.50,                              # Half to test, half to validation.
    random_state=SEED,                           # Repeatable split.
    stratify=[label for _, label in remaining]   # Keep class proportions equal.
)

print('Folder-to-label mapping:', catalog.class_to_idx)  # Which number each class received.
print('Train:', len(train_records), '| Validation:', len(val_records), '| Test:', len(test_records))  # Split sizes.

## 3. Image preprocessing, datasets, and loaders

As in the class notebook, validation and test images are only resized; training images are optionally augmented with a random horizontal flip, a rotation of up to ±15°, and a mild brightness/contrast change.

The training loader is built inside `run_experiment` (section 5) because it depends on whether the run uses augmentation. `test_loader` goes through **every** test image.

In [ ]:
# Deterministic preparation for validation and test images (and for non-augmented training).
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),   # Resize to 64 x 64.
    transforms.ToTensor(),                          # Convert to a [3, 64, 64] tensor with values 0..1.
])

# Random changes, applied ONLY to training images in the augmented runs.
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),   # Resize to 64 x 64.
    transforms.RandomHorizontalFlip(p=0.5),  # Mirror approximately half the photos.
    transforms.RandomRotation(15),          # Small rotations; corners may be padded black.
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Gentle lighting variation.
    transforms.ToTensor(),                  # Convert to [channels, height, width].
])

class PhotoDataset(Dataset):                # Loads photos from a list of (path, label) records.
    def __init__(self, records, transform):
        self.records = records              # Remember the (path, label) list.
        self.transform = transform          # Remember the preprocessing to apply.

    def __len__(self):
        return len(self.records)            # Number of photos in this dataset.

    def __getitem__(self, index):
        path, label = self.records[index]   # Look up one record.
        with Image.open(path) as picture:   # Open the photo file.
            # Convert even grayscale/RGBA files to three-channel RGB.
            image = self.transform(picture.convert('RGB'))  # Then apply the transform -> tensor.
        return image, label                 # One (image tensor, label) pair.

val_dataset = PhotoDataset(val_records, image_transform)    # Validation photos: resize only.
test_dataset = PhotoDataset(test_records, image_transform)  # Test photos: resize only.
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)    # Validation batches, fixed order.
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)  # Test batches, fixed order; covers ALL test photos.

## 4. The two networks: original `BasicCNN` and new `BatchNormCNN`

`BasicCNN` is copied unchanged from the class notebook. For one image the shapes are `3×64×64 → 16×32×32 → 32×16×16 → 64×8×8 → 4096 → 64 → 5`.

`BatchNormCNN` adds one `nn.BatchNorm2d` layer **after each `Conv2d`** (Conv → BatchNorm → ReLU → MaxPool). For every channel, BatchNorm rescales the convolution outputs in a batch to mean 0 and standard deviation 1, then applies a learned scale and shift. This usually makes training faster and more stable. The number given to `BatchNorm2d` is the number of channels coming out of the preceding `Conv2d` (16, 32, 64). In training mode (`model.train()`) it uses the statistics of the current batch; in evaluation mode (`model.eval()`) it uses running averages saved during training. Shapes do not change.

In [ ]:
class BasicCNN(nn.Module):                       # Original network from the class notebook (unchanged).
    def __init__(self, num_classes):
        super().__init__()                       # Set up the nn.Module machinery.
        self.features = nn.Sequential(           # Feature extractor: three Conv -> ReLU -> Pool blocks.
            nn.Conv2d(3, 16, kernel_size=3, padding=1),   # 3 color channels -> 16 feature maps, 3x3 filters.
            nn.ReLU(),                                    # Keep positives, zero out negatives.
            nn.MaxPool2d(2),                              # Halve height and width: 64 -> 32.
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16 -> 32 feature maps.
            nn.ReLU(),                                    # Nonlinearity.
            nn.MaxPool2d(2),                              # 32 -> 16.
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 32 -> 64 feature maps.
            nn.ReLU(),                                    # Nonlinearity.
            nn.MaxPool2d(2),                              # 16 -> 8.
        )
        self.classifier = nn.Sequential(         # Classifier: feature maps -> class scores.
            nn.Flatten(),                                 # 64 x 8 x 8 maps -> vector of 4096 numbers.
            nn.Linear(64 * (IMAGE_SIZE // 8) ** 2, 64),   # 4096 -> 64 hidden units.
            nn.ReLU(),                                    # Nonlinearity.
            nn.Linear(64, num_classes),                   # 64 -> one logit per class.
        )

    def forward(self, x):
        return self.classifier(self.features(x)) # Run the features, then the classifier.

class BatchNormCNN(nn.Module):                   # Problem 2: BasicCNN + BatchNorm2d after each Conv2d.
    def __init__(self, num_classes):
        super().__init__()                       # Set up the nn.Module machinery.
        self.features = nn.Sequential(           # Feature extractor: three Conv -> BatchNorm -> ReLU -> Pool blocks.
            nn.Conv2d(3, 16, kernel_size=3, padding=1),   # 3 color channels -> 16 feature maps.
            nn.BatchNorm2d(16),                           # NEW: normalize each of the 16 channels.
            nn.ReLU(),                                    # Nonlinearity.
            nn.MaxPool2d(2),                              # 64 -> 32.
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16 -> 32 feature maps.
            nn.BatchNorm2d(32),                           # NEW: normalize each of the 32 channels.
            nn.ReLU(),                                    # Nonlinearity.
            nn.MaxPool2d(2),                              # 32 -> 16.
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 32 -> 64 feature maps.
            nn.BatchNorm2d(64),                           # NEW: normalize each of the 64 channels.
            nn.ReLU(),                                    # Nonlinearity.
            nn.MaxPool2d(2),                              # 16 -> 8.
        )
        self.classifier = nn.Sequential(         # Same classifier as BasicCNN.
            nn.Flatten(),                                 # 64 x 8 x 8 -> 4096.
            nn.Linear(64 * (IMAGE_SIZE // 8) ** 2, 64),   # 4096 -> 64.
            nn.ReLU(),                                    # Nonlinearity.
            nn.Linear(64, num_classes),                   # 64 -> one logit per class.
        )

    def forward(self, x):
        return self.classifier(self.features(x)) # Run the features, then the classifier.

criterion = nn.CrossEntropyLoss()                # Loss for multi-class classification (same for every run).
print(BatchNormCNN(len(class_names)))            # Show the new network's layers.
for cnn_class in [BasicCNN, BatchNormCNN]:       # Compare the sizes of the two networks.
    print(cnn_class.__name__, 'trainable parameters:', sum(p.numel() for p in cnn_class(len(class_names)).parameters()))

## 5. Train one network and evaluate it on the entire test set

`evaluate` and the training loop are the class-notebook code. `run_experiment` wraps them so the same code can run for each network/augmentation combination:

1. Build the training loader (augmented or not) and a fresh network with the same seed.
2. Train for `EPOCHS` epochs and keep the weights with the lowest **validation loss**.
3. Predict **every** image in the test set and compute accuracy and **overall precision**: **macro** (the average of the five per-class precisions; each class counts equally) and **weighted** (each class weighted by its number of test images). Macro precision is the main number we compare.

In [ ]:
def evaluate(model, loader):                     # Average loss and accuracy over a whole loader.
    model.eval()                                 # Evaluation mode (BatchNorm uses running statistics).
    loss_sum, correct, count = 0.0, 0, 0         # Running totals.
    with torch.no_grad():                        # No gradients needed.
        for inputs, labels in loader:            # For each batch:
            inputs, labels = inputs.to(device), labels.to(device)  # move it to the device,
            logits = model(inputs)               # predict class scores,
            loss_sum += criterion(logits, labels).item() * len(labels)  # add the batch's total loss,
            correct += (logits.argmax(1) == labels).sum().item()   # count correct predictions,
            count += len(labels)                 # and count the images.
    return loss_sum / count, correct / count     # Mean loss and accuracy.

def run_experiment(cnn_class, use_augmentation): # Train one network with or without augmentation; return its results.
    name = (f"{'BatchNorm' if cnn_class is BatchNormCNN else 'Original'} CNN, "
            f"{'with' if use_augmentation else 'no'} augmentation")  # Label for printing and plots.
    print(f'\n========== {name} ==========')
    train_dataset = PhotoDataset(train_records, train_transform if use_augmentation else image_transform)  # Training photos.
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,       # Batches of 64 training photos,
                              shuffle=True,                               # in a new random order each epoch,
                              generator=torch.Generator().manual_seed(SEED),  # with a repeatable shuffle,
                              num_workers=0)                              # loaded in the main process.

    # Reset before model creation so every experiment starts from the same random state.
    torch.manual_seed(SEED)
    model = cnn_class(len(class_names)).to(device)   # Fresh network on the device.
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)  # Adam updates all weights.

    history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}  # Metrics per epoch.
    best_val_loss = float('inf')                 # Lowest validation loss so far.
    best_state = None                            # Weights at that epoch.
    start = time.perf_counter()                  # Start the timer.
    for epoch in range(1, EPOCHS + 1):           # Repeat for every epoch:
        model.train()                            # Training mode (BatchNorm uses batch statistics).
        loss_sum, correct, count = 0.0, 0, 0     # Running totals for this epoch.
        for inputs, labels in train_loader:      # For each training batch:
            inputs, labels = inputs.to(device), labels.to(device)  # Move the batch to the device.
            optimizer.zero_grad()             # Clear gradients from the last batch.
            logits = model(inputs)            # Forward pass.
            loss = criterion(logits, labels)  # Compare scores with true labels.
            loss.backward()                   # Calculate weight gradients.
            optimizer.step()                  # Update weights using Adam.
            loss_sum += loss.item() * len(labels)                  # Add the batch's total loss.
            correct += (logits.argmax(1) == labels).sum().item()   # Count correct predictions.
            count += len(labels)                                   # Count the images.
        train_loss, train_accuracy = loss_sum / count, correct / count  # Training metrics for this epoch.
        val_loss, val_accuracy = evaluate(model, val_loader)       # Validation metrics with fixed weights.
        for key, value in [('train_loss', train_loss), ('val_loss', val_loss),
                           ('train_accuracy', train_accuracy), ('val_accuracy', val_accuracy)]:
            history[key].append(value)           # Record all four numbers.
        if val_loss < best_val_loss:             # New best epoch on validation loss:
            best_val_loss, best_epoch = val_loss, epoch  # remember it,
            # A copy is necessary: a plain reference would keep changing during training.
            best_state = copy.deepcopy(model.state_dict())  # and keep a copy of its weights.
        print(f'Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.3f}, acc {train_accuracy:.1%}'
              f' | val loss {val_loss:.3f}, acc {val_accuracy:.1%}', flush=True)  # Progress line.
    model.load_state_dict(best_state)            # Go back to the best epoch's weights.
    print(f'Restored epoch {best_epoch}, selected using validation loss.')
    print(f'Training time: {(time.perf_counter() - start) / 60:.1f} minutes')

    # Evaluate on the ENTIRE test set.
    model.eval()                                 # Evaluation mode (BatchNorm uses running statistics).
    true_labels, predicted_labels = [], []       # Collected over the whole test set.
    with torch.no_grad():                        # No gradients needed.
        for inputs, labels in test_loader:       # Every test batch, so every test image:
            logits = model(inputs.to(device))    # predict class scores,
            predicted_labels.extend(logits.argmax(1).cpu().numpy())  # keep the predicted classes,
            true_labels.extend(labels.numpy())   # and the true classes.
    true_labels = np.asarray(true_labels)        # Lists -> NumPy arrays.
    predicted_labels = np.asarray(predicted_labels)
    assert len(true_labels) == len(test_dataset) # Confirm the entire test set was evaluated.
    result = {                                   # Everything needed for the comparison in section 7.
        'name': name,
        'best_epoch': best_epoch,
        'history': history,
        'test_accuracy': accuracy_score(true_labels, predicted_labels),  # Fraction correct.
        'precision_macro': precision_score(true_labels, predicted_labels, average='macro', zero_division=0),  # Overall precision, classes equal.
        'precision_weighted': precision_score(true_labels, predicted_labels, average='weighted', zero_division=0),  # Overall precision, weighted by class size.
        'precision_per_class': precision_score(true_labels, predicted_labels, average=None,
                                               labels=list(range(len(class_names))), zero_division=0),  # Precision of each class.
    }
    print(f'Test images evaluated: {len(true_labels)} of {len(test_dataset)}')  # Shows the full test set was used.
    print(f"Test accuracy: {result['test_accuracy']:.2%}")
    print(f"Overall precision (macro): {result['precision_macro']:.2%}")
    print(f"Overall precision (weighted): {result['precision_weighted']:.2%}")
    print(classification_report(true_labels, predicted_labels,   # Per-class precision, recall, F1, support.
          labels=list(range(len(class_names))), target_names=class_names, zero_division=0))
    return result                                # Hand the results back to the caller.

## 6. Run the four experiments

Each run trains a fresh network for 30 epochs, so this cell takes several minutes on a Colab GPU (longer on CPU). Use **Runtime → Change runtime type → GPU** in Colab.

In [ ]:
experiments = [                                  # (network, use augmentation?) for each of the four runs.
    (BasicCNN, False),                           # 1. Original network, no augmentation.
    (BasicCNN, True),                            # 2. Original network, with augmentation.
    (BatchNormCNN, False),                       # 3. BatchNorm network, no augmentation.
    (BatchNormCNN, True),                        # 4. BatchNorm network, with augmentation.
]
results = [run_experiment(cnn_class, use_augmentation)  # Train and test each one,
           for cnn_class, use_augmentation in experiments]  # keeping all four results in a list.

## 7. Compare precision of the original and BatchNorm networks

All numbers below are on the **entire test set** (551 images).

In [ ]:
print(f"{'Run':<40}{'Precision':>11}{'Precision':>12}{'Accuracy':>10}{'Best':>6}")    # Table header, line 1.
print(f"{'':<40}{'(macro)':>11}{'(weighted)':>12}{'':>10}{'epoch':>6}")              # Table header, line 2.
for r in results:                                # One row per run.
    print(f"{r['name']:<40}{r['precision_macro']:>11.2%}{r['precision_weighted']:>12.2%}"
          f"{r['test_accuracy']:>10.2%}{r['best_epoch']:>6}")

print('\nPrecision per class:')                  # Second table: precision of each flower class.
print(f"{'Run':<40}" + ''.join(f'{name:>12}' for name in class_names))  # Header with class names.
for r in results:                                # One row per run.
    print(f"{r['name']:<40}" + ''.join(f'{p:>12.2%}' for p in r['precision_per_class']))

fig, ax = plt.subplots(figsize=(10, 4))          # Bar chart of overall (macro) precision.
bars = ax.bar([r['name'].replace(', ', '\n') for r in results],  # One bar per run (label on two lines),
              [r['precision_macro'] for r in results],           # bar height = macro precision,
              color=['#9db4d3', '#4a78b5', '#e0a96d', '#c4661f'])  # light/dark blue = original, light/dark orange = BatchNorm.
ax.bar_label(bars, labels=[f"{r['precision_macro']:.1%}" for r in results])  # Write the value on each bar.
ax.set(ylabel='Overall precision (macro)', ylim=(0, 1),
       title='Overall precision on the full test set (551 images)')
ax.grid(axis='y', alpha=0.25)                    # Light horizontal grid.
plt.tight_layout()                               # Reduce overlap.
plt.show()                                       # Display the figure.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # Validation curves of all four runs, side by side.
for r in results:                                # One line per run in each plot.
    axes[0].plot(range(1, EPOCHS + 1), r['history']['val_loss'], label=r['name'])
    axes[1].plot(range(1, EPOCHS + 1), r['history']['val_accuracy'], label=r['name'])
axes[0].set(xlabel='Epoch', ylabel='Validation loss', title='Validation loss by epoch')
axes[1].set(xlabel='Epoch', ylabel='Validation accuracy', title='Validation accuracy by epoch', ylim=(0, 1))
for ax in axes:
    ax.legend(fontsize=8)                        # Legend in each plot.
    ax.grid(alpha=0.25)                          # Light grid.
plt.tight_layout()                               # Reduce overlap.
plt.show()                                       # Display the figure.

## 8. Report on results

_Fill in this section after running sections 6 and 7._

| Network | Augmentation | Overall precision (macro) | Overall precision (weighted) | Test accuracy | Best epoch |
|---|---|---|---|---|---|
| Original `BasicCNN` | No | | | | |
| Original `BasicCNN` | Yes | | | | |
| `BatchNormCNN` | No | | | | |
| `BatchNormCNN` | Yes | | | | |

Points to discuss:

1. **Effect of Batch Normalization without augmentation:** compare runs 1 and 3. Did precision go up or down, and by how much?
2. **Effect of Batch Normalization with augmentation:** compare runs 2 and 4.
3. **Effect of augmentation on each network:** compare runs 1 vs 2 and runs 3 vs 4.
4. **Training behaviour:** in the validation curves, did BatchNorm reach a low validation loss in fewer epochs (see the best epoch column)? Did any run overfit (validation loss rising while training continues)?
5. **Per-class precision:** which flower classes improved or got worse (for example roses vs tulips, which are often confused)?
6. **Caveat:** each number comes from one run with one seed on 551 test images, so differences of 1–2 percentage points may be noise rather than a real effect.